### Objective :-
##### Transform the raw datasets into clean, consistent, and reliable data.

#### Sections in this Notebook:-

1. Import Libraries

2. Load Raw Data

3. Create Data Copies for Cleaning

4. Initial Dataset Overview

5. Standardize Column Names

6. Duplicate Records Handling

7. Missing Value Analysis and Treatment

8. Data Type Conversion

9. Text Data Cleaning

10. Numerical Data Cleaning

11. Outlier Detection and Treatment

12. Data Validation Checks

13. Primary Key and Foreign Key Validation

14. Relationship Consistency Checks

15. Final Dataset Summary

16. Export Cleaned Datasets

17. Conclusion and Cleaning Summary

### Step 1: Import Libraries

In [34]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

### Step 2: Load Raw Data

In [35]:
apps = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\raw\apps.csv")
country = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\raw\app_country_stats.csv",low_memory=False)
discovery = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\raw\discovery_signals.csv",low_memory=False)

#### Creating copies so the originals remain unchanged:

In [36]:
apps_clean = apps.copy()
country_clean = country.copy()
discovery_clean = discovery.copy()

### Step 3: Check Dataset Shapes

In [37]:
print(apps_clean.shape)
print(country_clean.shape)
print(discovery_clean.shape)

(11176, 46)
(111307, 12)
(75812, 10)


### Step 4: Standardize Column Names

Converting column names to lowercase and replace spaces with underscores.

In [38]:
apps_clean.columns = (
    apps_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

country_clean.columns = (
    country_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

discovery_clean.columns = (
    discovery_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

### Step 5: Remove Exact Duplicates

In [39]:
apps_clean = apps_clean.drop_duplicates()

country_clean = country_clean.drop_duplicates()

discovery_clean = discovery_clean.drop_duplicates()

1. Number of duplicates removed
2. Whether they were complete duplicates or require further investigation

##### Duplicate Records Validation

Duplicate records were checked during the data understanding phase.
The same validation is performed again during cleaning to ensure data integrity
before further transformations.

Since no duplicate records were detected, no rows were removed.

### Step 6: Missing Value Analysis and Treatment

#### 6.1 Apps_clean (main)

#### 6.1 (a) Understanding Missing Values

In [40]:
apps_clean.isnull().sum()

app_id                            0
title                             0
description                       0
summary                           0
installs                         31
min_installs                     31
max_installs                      0
score                          2245
ratings                        2245
reviews                        2245
histogram                         0
price                            31
free                              0
currency                         31
sale                              0
sale_time                     11176
original_price                11176
developer                         0
developer_id                      0
developer_email                   0
developer_website              1176
developer_address             11175
privacy_policy                    2
genre                             0
genre_id                          0
categories                        0
icon                              2
header_image                

#### Columns with zero (0) missing values ✅

In [41]:
apps_clean.columns[apps_clean.isnull().sum() == 0]

Index(['app_id', 'title', 'description', 'summary', 'max_installs',
       'histogram', 'free', 'sale', 'developer', 'developer_id',
       'developer_email', 'genre', 'genre_id', 'categories', 'screenshots',
       'content_rating', 'ad_supported', 'contains_ads', 'in_app_purchases',
       'version', 'scraped_at'],
      dtype='object')

### Dropping redundant Columns:-
| Column                     | Reason                                                   |
| -------------------------- | -------------------------------------------------------- |
| `sale_time`                | Almost completely missing (11176)                        |
| `original_price`           | Almost completely missing (11176)                        |
| `developer_address`        | Almost completely missing (11175) and usually not useful |
| `video`                    | Mostly missing and harder to use                         |
| `video_image`              | Mostly missing                                           |
| `size`                     | Completely missing                                       |
| `android_version`          | Completely missing                                       |
| `android_version_text`     | Completely missing                                       |
| `developer_internal_id`    | Internal identifier, usually not useful for analysis     |
| `required_android_version` | Completely missing                                       |
| `interactive_elements`     | Completely missing                                       |
| `recent_changes`           | Completely missing                                       |


In [42]:
drop_cols = [
    "sale_time",
    "original_price",
    "developer_address",
    "video",
    "video_image",
    "size",
    "android_version",
    "android_version_text",
    "developer_internal_id",
    "required_android_version",
    "interactive_elements",
    "recent_changes"
]

apps_clean = apps_clean.drop(columns=drop_cols)

In [43]:
apps_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11176 entries, 0 to 11175
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   app_id                      11176 non-null  object 
 1   title                       11176 non-null  object 
 2   description                 11176 non-null  object 
 3   summary                     11176 non-null  object 
 4   installs                    11145 non-null  object 
 5   min_installs                11145 non-null  float64
 6   max_installs                11176 non-null  int64  
 7   score                       8931 non-null   float64
 8   ratings                     8931 non-null   float64
 9   reviews                     8931 non-null   float64
 10  histogram                   11176 non-null  object 
 11  price                       11145 non-null  float64
 12  free                        11176 non-null  bool   
 13  currency                    111

In [44]:
apps_clean.dtypes

app_id                         object
title                          object
description                    object
summary                        object
installs                       object
min_installs                  float64
max_installs                    int64
score                         float64
ratings                       float64
reviews                       float64
histogram                      object
price                         float64
free                             bool
currency                       object
sale                             bool
developer                      object
developer_id                   object
developer_email                object
developer_website              object
privacy_policy                 object
genre                          object
genre_id                       object
categories                     object
icon                           object
header_image                   object
screenshots                    object
content_rati

#### Handle remaining missing values

In [45]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

installs                        31
min_installs                    31
score                         2245
ratings                       2245
reviews                       2245
price                           31
currency                        31
developer_website             1176
privacy_policy                   2
icon                             2
header_image                     2
content_rating_description    9909
updated                       3394
dtype: int64

##### 1. installs, min_installs (31 missing) → very small.

In [51]:
apps_clean.loc[
    apps_clean["installs"].isnull(),
    ["title", "installs", "min_installs", "max_installs"]
]

,title,installs,min_installs,max_installs
7298,Budget Planner - Money Manager,NaN,NaN,0
7635,Meow Knights: Idle RPG,NaN,NaN,0
7683,Toy Survivor : The War,NaN,NaN,0
7684,Servant of the Lake,NaN,NaN,2
7685,Evil Sword,NaN,NaN,4
7686,FantasyTown,NaN,NaN,932644
7687,DIGIMON UP,NaN,NaN,2
7688,Merge City - Travel & Story,NaN,NaN,0
7689,Stillpoint: Minimal Defense,NaN,NaN,2
7690,"Supremacy: Warhammer 40,000",NaN,NaN,199108


### Handling Missing Values in Install Data

The columns `installs` and `min_installs` contain 31 missing values each. Since install count is an important metric for app popularity analysis, dropping these rows would result in unnecessary data loss.

The dataset already provides `min_installs` and `max_installs`, which represent the possible range of installations for an app. Therefore, we create a new feature called `estimated_installs` by calculating the midpoint between these two values:

\[
estimated\_installs = {min\_installs + max\_installs} / 2
\]

This estimated value is then used to fill the missing values in the `installs` column.

In [52]:
# Create estimated installs
apps_clean["estimated_installs"] = (
    apps_clean["min_installs"] + apps_clean["max_installs"]
) / 2

# For rows where estimated_installs is NaN (because min_installs is NaN),
# use max_installs as the estimate.
apps_clean["estimated_installs"] = apps_clean["estimated_installs"].fillna(
    apps_clean["max_installs"]
)

# Fill missing installs
apps_clean["installs"] = apps_clean["installs"].fillna(
    apps_clean["estimated_installs"]
)

# Fill missing min_installs
apps_clean["min_installs"] = apps_clean["min_installs"].fillna(
    apps_clean["estimated_installs"]
)

#### Checking Missing Values

After filling the missing values, we verify that the `installs` and `min_installs` columns no longer contain null values.

In [53]:
apps_clean[["installs", "min_installs"]].isnull().sum()

installs        0
min_installs    0
dtype: int64

In [54]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

score                         2245
ratings                       2245
reviews                       2245
price                           31
currency                        31
developer_website             1176
privacy_policy                   2
icon                             2
header_image                     2
content_rating_description    9909
updated                       3394
dtype: int64

##### 2. score, ratings, reviews (2245 missing)

These three columns are closely related. Before filling them, verify that missing values correspond to apps with no ratings.

In [55]:
apps_clean.loc[
    apps_clean["score"].isnull(),
    ["title", "score", "ratings", "reviews"]
].head()

,title,score,ratings,reviews
96,Budgey: Cashback & Vouchers,NaN,NaN,NaN
246,Resume Builder: CV Maker,NaN,NaN,NaN
507,Akinfo - Repair Tools & Parts,NaN,NaN,NaN
609,"DAS MEI: Boleto, INSS, Imposto",NaN,NaN,NaN
961,Spoken English through Hindi,NaN,NaN,NaN


If all three are missing together (which is common for newly launched apps), fill them with 0:

In [56]:
apps_clean["score"] = apps_clean["score"].fillna(0)
apps_clean["ratings"] = apps_clean["ratings"].fillna(0)
apps_clean["reviews"] = apps_clean["reviews"].fillna(0)

##### Explanation:

The score, ratings, and reviews columns had missing values for the same set of apps. This generally indicates that these apps have not yet received user ratings or reviews. Therefore, the missing values were replaced with 0 to represent the absence of user feedback.

##### 3. price and currency (31 missing)

Since we have a free column, first check whether these missing values belong to free apps:

In [57]:
apps_clean.loc[
    apps_clean["price"].isnull(),
    ["title", "free", "price", "currency"]
]

,title,free,price,currency
7298,Budget Planner - Money Manager,False,NaN,NaN
7635,Meow Knights: Idle RPG,False,NaN,NaN
7683,Toy Survivor : The War,False,NaN,NaN
7684,Servant of the Lake,False,NaN,NaN
7685,Evil Sword,False,NaN,NaN
7686,FantasyTown,False,NaN,NaN
7687,DIGIMON UP,False,NaN,NaN
7688,Merge City - Travel & Story,False,NaN,NaN
7689,Stillpoint: Minimal Defense,False,NaN,NaN
7690,"Supremacy: Warhammer 40,000",False,NaN,NaN


We found none of these apps are free, so we have to look at other sale-related columns.

In [60]:
apps_clean.loc[
    apps_clean["price"].isnull(),
    ["title", "free", "sale", "price", "currency"]
]

,title,free,sale,price,currency
7298,Budget Planner - Money Manager,False,False,NaN,NaN
7635,Meow Knights: Idle RPG,False,False,NaN,NaN
7683,Toy Survivor : The War,False,False,NaN,NaN
7684,Servant of the Lake,False,False,NaN,NaN
7685,Evil Sword,False,False,NaN,NaN
7686,FantasyTown,False,False,NaN,NaN
7687,DIGIMON UP,False,False,NaN,NaN
7688,Merge City - Travel & Story,False,False,NaN,NaN
7689,Stillpoint: Minimal Defense,False,False,NaN,NaN
7690,"Supremacy: Warhammer 40,000",False,False,NaN,NaN


##### Imputing Missing Price Values

The `price` column contains some missing values. Since app prices can vary depending on the app's genre, we can use the genre information to estimate the missing prices.

Instead of directly assigning a single value to all missing prices, we will calculate the typical price of each genre and use it for imputation.

However, before choosing the appropriate statistical measure (mean, median, or mode), we need to analyze the price distribution and check for the presence of outliers or skewness in the data.

- **Mean** can be affected by extreme values (outliers).
- **Median** is more robust and can represent the typical price better when the data is skewed.
- **Mode** can be useful if there is a most frequently occurring price value.

Therefore, we will first analyze the distribution of prices within each genre and then select the most appropriate method for filling the missing values.

In [61]:
apps_clean.groupby("genre")["price"].describe()

,count,mean,std,min,25%,50%,75%,max
genre,,,,,,,,
Action,152.0,0.118158,0.889669,0.0,0.0,0.0,0.00,9.99
Adventure,52.0,2.766538,6.541912,0.0,0.0,0.0,1.49,29.99
Arcade,75.0,0.339200,1.346223,0.0,0.0,0.0,0.00,6.99
Art & Design,176.0,0.113295,0.671210,0.0,0.0,0.0,0.00,4.99
Auto & Vehicles,35.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.00
Beauty,87.0,0.045862,0.427773,0.0,0.0,0.0,0.00,3.99
Board,122.0,0.110410,0.707066,0.0,0.0,0.0,0.00,4.99
Books & Reference,237.0,0.088481,0.879919,0.0,0.0,0.0,0.00,11.99
Business,506.0,0.158992,2.239099,0.0,0.0,0.0,0.00,39.99


In [62]:
apps_clean["price"].skew()

20.25019079072559

##### Choosing the Appropriate Imputation Method for Missing Prices

The missing values in the `price` column cannot be filled directly because app prices vary across different genres. Therefore, genre-based imputation was considered.

Before selecting an imputation method, the distribution of prices was analyzed to determine whether mean, median, or mode would be more appropriate.

The overall skewness of the `price` column was calculated:

**Skewness = 20.25**

This indicates a highly right-skewed distribution, where most apps have low prices (mostly 0) and a small number of expensive apps increase the average price significantly.

Additionally, the genre-wise price statistics showed that the median price for most genres is 0, while the mean price is higher due to the presence of expensive outliers.

Since the mean is sensitive to outliers, using it would overestimate missing prices. Therefore, the median price of each genre was selected for imputing missing values because it represents the typical price of apps within that genre and is more robust to extreme values.

In [64]:
# Calculate genre-wise median price
genre_median_price = apps_clean.groupby("genre")["price"].transform("median")

# Fill missing prices using genre median
apps_clean["price"] = apps_clean["price"].fillna(genre_median_price)

In [65]:
apps_clean[apps_clean["price"].isnull()][["title", "genre", "price"]]

,title,genre,price


In [66]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

currency                        31
developer_website             1176
privacy_policy                   2
icon                             2
header_image                     2
content_rating_description    9909
updated                       3394
dtype: int64

#### Now Checking For Currency 

In [67]:
apps_clean["currency"].value_counts(dropna=False)

currency
USD    11072
INR       72
NaN       31
GBP        1
Name: count, dtype: int64

##### Handling Missing Currency Values

The `currency` column contains 31 missing values. Since currency is a categorical variable, numerical imputation methods such as mean or median cannot be applied.

The distribution of available currency values was analyzed:

- USD: 11,072 records
- INR: 72 records
- GBP: 1 record

USD is the dominant currency in the dataset:

\[
\frac{11072}{11072 + 72 + 1} \times 100 \approx 99.3\%
\]

Therefore, USD represents approximately **99.3% of all available currency values**, making it the most representative value for imputing the missing currency entries.

Since USD represents approximately 99.3% of all non-missing currency values, it is reasonable to assume that the missing currency values most likely belong to the same category.

Additionally, these missing currency values occur in the same records where the `price` column was missing. Since the missing prices were already imputed based on genre-specific median prices, the corresponding currency values were filled using the most frequent currency value (mode).

Therefore, mode imputation was selected for handling missing values in the `currency` column.

In [68]:
# Filling missing currency values with the most frequent currency
apps_clean["currency"] = apps_clean["currency"].fillna(
    apps_clean["currency"].mode()[0]
)

In [69]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

developer_website             1176
privacy_policy                   2
icon                             2
header_image                     2
content_rating_description    9909
updated                       3394
dtype: int64

##### 4 : Handling Missing Developer Website Values (1176 missing values)

The `developer_website` column contains 1176 missing values. Removing these rows would lead to unnecessary data loss because the missing website information does not affect other important app attributes.

Since a developer website is optional metadata, the missing values were replaced with `"Not Available"`. This allows us to retain all app records while clearly indicating that website information was not provided.

In [70]:
apps_clean["developer_website"] = apps_clean["developer_website"].fillna(
    "Not Available"
)

In [71]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

privacy_policy                   2
icon                             2
header_image                     2
content_rating_description    9909
updated                       3394
dtype: int64

##### 5: Handling Missing App Resource Information

The columns `privacy_policy`, `icon`, and `header_image` contain only 2 missing values each.

Although the number of missing values is very small, removing these records would result in unnecessary data loss because the missing information only represents unavailable resources rather than incomplete app records.

These columns act as availability indicators:

- Missing `privacy_policy` indicates that a privacy policy link was not provided.
- Missing `icon` indicates that the app icon image is unavailable.
- Missing `header_image` indicates that the header image is unavailable.

Therefore, the missing values were replaced with `"Not Available"` to retain all records while preserving information about the absence of these resources.

In [72]:
apps_clean["privacy_policy"] = apps_clean["privacy_policy"].fillna(
    "Not Available"
)

apps_clean["icon"] = apps_clean["icon"].fillna(
    "Not Available"
)

apps_clean["header_image"] = apps_clean["header_image"].fillna(
    "Not Available"
)

In [73]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

content_rating_description    9909
updated                       3394
dtype: int64

##### 5. Content_rating_description (9909 missing)

In [74]:
apps_clean[
    ["content_rating", "content_rating_description"]
].drop_duplicates()

,content_rating,content_rating_description
0,Everyone,NaN
1,Teen,Diverse Content: Discretion Advised
2,Teen,NaN
14,Everyone 10+,Diverse content: Discretion advised
39,Mature 17+,NaN
...,...,...
10392,Teen,"Violence, Blood, Use of Tobacco"
10393,Teen,"Mild Violence, Mild Blood, Suggestive Themes, ..."
10546,Teen,"Suggestive Themes, Language, Use of Tobacco"
10673,Everyone 10+,"Fantasy Violence, Tobacco Reference"


In [75]:
apps_clean["content_rating_description"].nunique()

225

##### Handling Content Rating Description

The `content_rating_description` column contains 9909 missing values out of approximately 11176 records, meaning around 88.6% of the data is unavailable.

Although this column contains text and could theoretically be used for NLP tasks, the extremely high missing rate and the short nature of the available descriptions limit its usefulness for meaningful text analysis.

Additionally, the dataset already contains richer text-based features such as `description` and `summary`, which are more suitable for NLP applications.

After evaluating its contribution, this feature was considered less informative and removed from the dataset.

In [76]:
apps_clean = apps_clean.drop(
    columns=["content_rating_description"]
)

In [77]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

updated    3394
dtype: int64

##### 6. Handling `updated` (3394 missing)

In [79]:
#"Check Values"
apps_clean["updated"].head()

0    1.776860e+09
1    1.776151e+09
2    1.777555e+09
3    1.754659e+09
4    1.777974e+09
Name: updated, dtype: float64

In [80]:
apps_clean["updated"].dtype

dtype('float64')

In [82]:
# Convert to date time :-
apps_clean["updated"] = pd.to_datetime(
    apps_clean["updated"],
    errors="coerce"
)

In [83]:
# Check Data Range
apps_clean["updated"].describe()

count                             7782
mean     1970-01-01 00:00:01.768069259
min      1970-01-01 00:00:01.409857010
25%      1970-01-01 00:00:01.768530247
50%      1970-01-01 00:00:01.776249300
75%      1970-01-01 00:00:01.777472089
max      1970-01-01 00:00:01.778356582
Name: updated, dtype: object

##### Creating useful features:-

In [86]:
# Extract year for visualization
apps_clean["update_year"] = apps_clean["updated"].dt.year

In [87]:
# Extract month for visualization
apps_clean["update_month"] = apps_clean["updated"].dt.month

In [88]:
# Calculate recency
reference_date = pd.Timestamp.today()

apps_clean["days_since_update"] = (
    reference_date - apps_clean["updated"]
).dt.days

In [89]:
apps_clean["update_missing"] = apps_clean["updated"].isnull().astype(int)

##### Feature Extraction from Updated Date

The `updated` column represents the last update date of an application. Instead of removing this feature due to missing values, additional time-based features were extracted to make it more useful for analysis.

The following features were created:

- `update_year`: Helps analyze yearly update trends.
- `update_month`: Helps identify monthly update patterns.
- `days_since_update`: Measures how recently an app was updated.

These features provide flexibility for both visualization tools such as Power BI and analytical queries using SQL.

In [91]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

updated              3394
update_year          3394
update_month         3394
days_since_update    3394
dtype: int64

In [92]:
apps_clean["update_missing"] = apps_clean["updated"].isnull().astype(int)

In [93]:
apps_clean.isnull().sum()[apps_clean.isnull().sum() > 0]

updated              3394
update_year          3394
update_month         3394
days_since_update    3394
dtype: int64

##### Handling Missing Update Date Information

The `updated` column contains 3394 missing values. Since the exact update dates cannot be determined from the available data, these missing values were not replaced with artificial dates or estimated values.

The derived features `update_year`, `update_month`, and `days_since_update` also contain the same missing values because they are calculated directly from the `updated` column. If the original update date is unavailable, these features cannot be accurately generated.

Instead of imputing these values, the missing information was preserved to avoid introducing incorrect data. A separate indicator feature, `update_missing`, was created to capture whether an app has missing update information.

- `update_missing = 0` → Update date is available.
- `update_missing = 1` → Update date is unavailable.

This approach preserves the original data quality while allowing further analysis of whether missing update information itself has any relationship with app performance.

#### 6.2 Country_clean 

1. First check structure and missing values

In [94]:
country_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111307 entries, 0 to 111306
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            111307 non-null  int64  
 1   app_id        111307 non-null  object 
 2   country       111307 non-null  object 
 3   installs      110988 non-null  object 
 4   min_installs  110988 non-null  float64
 5   score         73811 non-null   float64
 6   ratings       73811 non-null   float64
 7   reviews       73811 non-null   float64
 8   price         110988 non-null  float64
 9   free          111306 non-null  object 
 10  scrape_date   111307 non-null  object 
 11  scraped_at    111307 non-null  object 
dtypes: float64(5), int64(1), object(6)
memory usage: 10.2+ MB


In [95]:
country_clean.isnull().sum()

id                  0
app_id              0
country             0
installs          319
min_installs      319
score           37496
ratings         37496
reviews         37496
price             319
free                1
scrape_date         0
scraped_at          0
dtype: int64

##### 1. installs and min_installs (319 missing)

In [96]:
country_clean[
    country_clean["installs"].isnull()
][["app_id", "country", "installs", "min_installs"]]

,app_id,country,installs,min_installs
1441,com.tet.thebudgetingapp.thebudgeting,us,NaN,NaN
1562,com.tet.thebudgetingapp.thebudgeting,in,NaN,NaN
1624,com.tet.thebudgetingapp.thebudgeting,br,NaN,NaN
1688,com.tet.thebudgetingapp.thebudgeting,id,NaN,NaN
2042,com.tet.thebudgetingapp.thebudgeting,mx,NaN,NaN
...,...,...,...,...
109709,com.playstation.mlbtheshowmobile,br,NaN,NaN
109710,com.playstation.mlbtheshowmobile,id,NaN,NaN
109712,com.playstation.mlbtheshowmobile,gb,NaN,NaN
109713,com.playstation.mlbtheshowmobile,de,NaN,NaN


Since our apps_clean dataset already has:

app_id
installs
min_installs
max_installs

we can use that!

This is actually better than estimating.

We can merge the app-level install information into country data:

In [97]:
country_clean = country_clean.merge(
    apps_clean[["app_id", "installs", "min_installs"]],
    on="app_id",
    how="left",
    suffixes=("", "_app")
)

In [98]:
country_clean["installs"] = country_clean["installs"].fillna(
    country_clean["installs_app"]
)

country_clean["min_installs"] = country_clean["min_installs"].fillna(
    country_clean["min_installs_app"]
)

In [99]:
country_clean.isnull().sum()[country_clean.isnull().sum() > 0]

score      37496
ratings    37496
reviews    37496
price        319
free           1
dtype: int64

##### 2. price (319 missing)

Same situation.

In [100]:
country_clean[
    country_clean["price"].isnull()
][["app_id", "country", "price"]]

,app_id,country,price
1441,com.tet.thebudgetingapp.thebudgeting,us,NaN
1562,com.tet.thebudgetingapp.thebudgeting,in,NaN
1624,com.tet.thebudgetingapp.thebudgeting,br,NaN
1688,com.tet.thebudgetingapp.thebudgeting,id,NaN
2042,com.tet.thebudgetingapp.thebudgeting,mx,NaN
...,...,...,...
109709,com.playstation.mlbtheshowmobile,br,NaN
109710,com.playstation.mlbtheshowmobile,id,NaN
109712,com.playstation.mlbtheshowmobile,gb,NaN
109713,com.playstation.mlbtheshowmobile,de,NaN


In [101]:
country_clean = country_clean.merge(
    apps_clean[["app_id", "price"]],
    on="app_id",
    how="left",
    suffixes=("", "_app")
)

In [102]:
country_clean["price"] = country_clean["price"].fillna(
    country_clean["price_app"]
)

In [103]:
country_clean.isnull().sum()[country_clean.isnull().sum() > 0]


score      37496
ratings    37496
reviews    37496
free           1
dtype: int64

##### 3. score, ratings, reviews (37496 missing)

This requires more investigation.

In [105]:
# Find Percentage
37496 / len(country_clean) * 100

33.68700980171957

##### Handling Missing App Performance Metrics

The columns `score`, `ratings`, and `reviews` contain 37,496 missing values, representing approximately 33.69% of the dataset.

Instead of applying statistical imputation methods such as mean or median, these values were investigated based on the relationship between `country_clean` and `apps_clean`.

Since `score`, `ratings`, and `reviews` are app-level attributes, the missing values can be recovered by matching `app_id` with the main app dataset.

Therefore, the missing values were filled using the corresponding values from `apps_clean`, avoiding artificial estimation and preserving the original app-level information.

In [106]:
# Check if app_id exists in apps_clean
country_clean["app_id"].isin(apps_clean["app_id"]).mean()

1.0

As, result is 1, this means that every country record has a matching app record.

In [107]:
country_clean = country_clean.merge(
    apps_clean[["app_id", "score", "ratings", "reviews"]],
    on="app_id",
    how="left",
    suffixes=("", "_app")
)

In [108]:
# Fill
country_clean["score"] = country_clean["score"].fillna(
    country_clean["score_app"]
)

country_clean["ratings"] = country_clean["ratings"].fillna(
    country_clean["ratings_app"]
)

country_clean["reviews"] = country_clean["reviews"].fillna(
    country_clean["reviews_app"]
)

In [109]:
# Remove helper columns:
country_clean = country_clean.drop(
    columns=["score_app", "ratings_app", "reviews_app"]
)

In [110]:
country_clean.isnull().sum()

id                  0
app_id              0
country             0
installs            0
min_installs        0
score               0
ratings             0
reviews             0
price               0
free                1
scrape_date         0
scraped_at          0
installs_app        0
min_installs_app    0
price_app           0
dtype: int64

4: Handling `free` 

In [111]:
country_clean[country_clean["free"].isnull()]

,id,app_id,country,installs,min_installs,score,ratings,reviews,price,free,scrape_date,scraped_at,installs_app,min_installs_app,price_app
37580,37460,com.carxtech.sr,mx,"10,000,000+",10000000.0,4.557769,611123.0,7565.0,0.0,NaN,2026-05-07,2026-05-07 10:15:31.108262+05:30,"10,000,000+",10000000.0,0.0


In [112]:
country_clean = country_clean.merge(
    apps_clean[["app_id", "free"]],
    on="app_id",
    how="left",
    suffixes=("", "_app")
)

country_clean["free"] = country_clean["free"].fillna(
    country_clean["free_app"]
)

country_clean = country_clean.drop(
    columns=["free_app"]
)

C:\Users\cashk\AppData\Local\Temp\ipykernel_4116\2098819088.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_clean["free"] = country_clean["free"].fillna(


##### Handling Missing Free Status

The `free` column contains only one missing value. Since the free/paid status is an app-level property and remains the same across different countries, the missing value was not imputed using statistical methods.

Instead, the corresponding value was retrieved from the main app dataset (`apps_clean`) using `app_id` as the key. This ensures consistency between the app-level and country-level datasets.

#### 6.3 Discovery_clean

##### 1. Checking structure and missing values

In [116]:
discovery_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75812 entries, 0 to 75811
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             75812 non-null  int64  
 1   app_id         75812 non-null  object 
 2   source         75812 non-null  object 
 3   country        71920 non-null  object 
 4   category       18726 non-null  object 
 5   keyword        53194 non-null  object 
 6   collection     18726 non-null  object 
 7   chart_rank     71920 non-null  float64
 8   signal_key     75812 non-null  object 
 9   discovered_at  75812 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 5.8+ MB


In [117]:
discovery_clean.isnull().sum()

id                   0
app_id               0
source               0
country           3892
category         57086
keyword          22618
collection       57086
chart_rank        3892
signal_key           0
discovered_at        0
dtype: int64

##### 1. id (0 missing)

In [118]:
discovery_clean["id"].nunique()

75812

##### 2. country (3892 missing)

3892 / 75812 ≈ 5.1%

In [119]:
discovery_clean[
    discovery_clean["country"].isnull()
].head()

,id,app_id,source,country,category,keyword,collection,chart_rank,signal_key,discovered_at
1206,147009,com.mandir,archive,NaN,NaN,NaN,NaN,NaN,com.mandir|archive|||||,2026-05-05 20:01:44.927620+05:30
1746,178849,com.mico,archive,NaN,NaN,NaN,NaN,NaN,com.mico|archive|||||,2026-05-05 20:03:44.579131+05:30
13193,72311,com.struckd.game,archive,NaN,NaN,NaN,NaN,NaN,com.struckd.game|archive|||||,2026-05-05 19:57:23.709412+05:30
13404,72312,com.cbn.mycbn.app,archive,NaN,NaN,NaN,NaN,NaN,com.cbn.mycbn.app|archive|||||,2026-05-05 19:57:23.712747+05:30
19754,72627,se.maginteractive.wordbrain,archive,NaN,NaN,NaN,NaN,NaN,se.maginteractive.wordbrain|archive|||||,2026-05-05 19:57:24.468287+05:30


In [120]:
discovery_clean["country"] = discovery_clean["country"].fillna(
    "Unknown"
)

##### Handling Missing Country Information

The `country` column contains missing values because some discovery records are not associated with a specific country.

Since these records still contain useful information about app discovery, the rows were retained and missing country values were replaced with `"Unknown"`.

##### 3. category (57086 missing)

57086 / 75812 ≈ 75%

In [121]:
discovery_clean.groupby("source")["category"].count()

source
archive           0
chart_page    18726
search            0
Name: category, dtype: int64

In [122]:
discovery_clean["category"] = discovery_clean["category"].fillna(
    "Not Available"
)

##### Handling Missing Category Information

The `category` column contains missing values because category information is only available for apps discovered through `chart_page` sources.

Records from `search` and `archive` sources do not have an associated chart category, therefore these missing values represent unavailable information rather than incorrect data.

Instead of removing these records, missing category values were replaced with `"Not Available"` to preserve all discovery records while maintaining the meaning of the data.

##### 4. Collection (57086 missing)

In [124]:
discovery_clean.groupby("source")["collection"].count()

source
archive           0
chart_page    18726
search            0
Name: collection, dtype: int64

In [125]:
discovery_clean["collection"] = discovery_clean["collection"].fillna(
    "Not Available"
)

##### 5. Keyword (22618 missing)

In [126]:
discovery_clean["keyword"] = discovery_clean["keyword"].fillna(
    "Not Available"
)

##### 6. Chart_rank (3892 missing)

In [127]:
discovery_clean["chart_rank_missing"] = (
    discovery_clean["chart_rank"].isnull().astype(int)
)

##### 7. Signal_key

In [128]:
discovery_clean["signal_key"].nunique()

75812

In [129]:
discovery_clean = discovery_clean.drop(
    columns=["signal_key"]
)

Reason:

It duplicates information already available in:

`app_id`,
`source`,
`country`,
`category`,
`collection`

##### 8. discovered_at

In [131]:
# Convert it to :
discovery_clean["discovered_at"] = pd.to_datetime(
    discovery_clean["discovered_at"],
    errors="coerce"
)

In [132]:
#Creating Useful Features
discovery_clean["discovery_year"] = (
    discovery_clean["discovered_at"].dt.year
)

discovery_clean["discovery_month"] = (
    discovery_clean["discovered_at"].dt.month
)

discovery_clean["discovery_hour"] = (
    discovery_clean["discovered_at"].dt.hour
)

In [133]:
discovery_clean.isnull().sum()

id                       0
app_id                   0
source                   0
country                  0
category                 0
keyword                  0
collection               0
chart_rank            3892
discovered_at            0
chart_rank_missing       0
discovery_year           0
discovery_month          0
discovery_hour           0
dtype: int64

##### Creating a missing indicator

In [134]:
discovery_clean["chart_rank_missing"] = (
    discovery_clean["chart_rank"].isnull().astype(int)
)

##### Handling Missing Chart Rank Information

The `chart_rank` column contains 3892 missing values. These missing values occur because not all discovery sources provide ranking information.

Chart ranks are only available for apps discovered through ranking-based sources such as `chart_page`. Other discovery methods, such as search or archive, do not have a ranking position.

Therefore, the missing values were not imputed because assigning artificial ranks would introduce incorrect information. Instead, a separate indicator feature (`chart_rank_missing`) was created to capture whether ranking information was available.

###  Validation

In [135]:
apps_clean.info()
country_clean.info()
discovery_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11176 entries, 0 to 11175
Data columns (total 38 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   app_id              11176 non-null  object        
 1   title               11176 non-null  object        
 2   description         11176 non-null  object        
 3   summary             11176 non-null  object        
 4   installs            11176 non-null  object        
 5   min_installs        11176 non-null  float64       
 6   max_installs        11176 non-null  int64         
 7   score               11176 non-null  float64       
 8   ratings             11176 non-null  float64       
 9   reviews             11176 non-null  float64       
 10  histogram           11176 non-null  object        
 11  price               11176 non-null  float64       
 12  free                11176 non-null  bool          
 13  currency            11176 non-null  object    

### Relationship Check

In [136]:
apps_clean.shape, country_clean.shape, discovery_clean.shape

((11176, 38), (111307, 15), (75812, 13))

In [137]:
country_clean["app_id"].isin(apps_clean["app_id"]).mean()

1.0

In [138]:
discovery_clean["app_id"].isin(apps_clean["app_id"]).mean()

0.9986545665593838

In [139]:
missing_discovery_apps = discovery_clean[
    ~discovery_clean["app_id"].isin(apps_clean["app_id"])
]

missing_discovery_apps.shape

(102, 13)

In [140]:
missing_discovery_apps.head()

,id,app_id,source,country,category,keyword,collection,chart_rank,discovered_at,chart_rank_missing,discovery_year,discovery_month,discovery_hour
3903,3882,com.zhiliaoapp.musically.go,chart_page,br,SOCIAL,Not Available,topselling_free,1.0,2026-05-05 01:25:41.814801+05:30,0,2026,5,1
3929,3908,com.zhiliaoapp.musically.go,chart_page,br,SOCIAL,Not Available,topselling_paid,1.0,2026-05-05 01:26:10.464384+05:30,0,2026,5,1
3962,3942,com.zhiliaoapp.musically.go,chart_page,br,SOCIAL,Not Available,topgrossing,3.0,2026-05-05 01:26:27.599519+05:30,0,2026,5,1
4158,4140,com.mobile.legends,chart_page,br,GAME,Not Available,topselling_free,16.0,2026-05-05 01:28:51.586998+05:30,0,2026,5,1
4203,4185,com.mobile.legends,chart_page,br,GAME,Not Available,topselling_paid,16.0,2026-05-05 01:29:13.238668+05:30,0,2026,5,1


In [141]:
missing_discovery_apps["app_id"].nunique()

17

In [142]:
len(missing_discovery_apps) / len(discovery_clean) * 100

0.13454334406162613

##### Handling Discovery Records Without Matching App Information

A small proportion of records in the `discovery_clean` dataset do not have a corresponding `app_id` match in the `apps_clean` dataset.

These records were not removed because the discovery dataset represents **app discovery events**, which are valuable independently from the main app information table.

The missing matches can occur because:
- An app may have been discovered during data collection but is no longer available in the main app dataset.
- The discovery dataset and app dataset may have been collected at different times.
- Some apps may exist only in discovery sources but not in the final app catalogue.

Removing these records would result in the loss of historical discovery information and could introduce bias by excluding certain apps or discovery sources.

Therefore, these records were retained. During SQL database modeling, `app_id` will be treated as a relationship key, and unmatched discovery records will remain available for discovery-level analysis while app-level details will only be available for matched applications.

In [143]:
apps_clean.to_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\apps_clean.csv", index=False)

country_clean.to_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\country_clean.csv", index=False)

discovery_clean.to_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\discovery_clean.csv", index=False)

In [144]:
# Verifying Saved Files:-
import os

os.getcwd()

'C:\\Users\\cashk\\OneDrive\\Desktop\\Projects\\Google_Playstore_project\\Notebooks'

In [146]:
os.chdir(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned")

In [147]:
os.getcwd()

'C:\\Users\\cashk\\OneDrive\\Desktop\\Projects\\Google_Playstore_project\\Playstore_Data\\cleaned'

In [148]:
os.listdir()

['apps_clean.csv', 'country_clean.csv', 'discovery_clean.csv']

# Final Data Cleaning Summary

The raw Google Play Store datasets (`apps`, `country`, and `discovery`) were cleaned and prepared for further analysis, SQL database creation, and Power BI visualization.

## 1. Apps Dataset (`apps_clean`)

The app-level dataset was cleaned by performing the following steps:

- Standardized column names by:
  - Removing leading/trailing spaces.
  - Converting column names to lowercase.
  - Replacing spaces with underscores.

- Removed duplicate records to ensure each application record was unique.

- Handled missing values based on the importance and meaning of each column:
  - Critical numerical features such as `installs` and `min_installs` were handled by creating an `estimated_installs` feature instead of removing records.
  - Missing `price` values were investigated and filled using genre-based price information.
  - Missing `currency` values were filled using mode imputation because currency is a categorical variable and USD represented the dominant pattern.
  - Optional information columns such as `developer_website`, `privacy_policy`, `icon`, and `header_image` were filled with `"Not Available"` to preserve records.
  - `content_rating_description` was removed because it had a very high percentage of missing values and provided limited additional information compared to `content_rating`.
  - Date-related missing values in `updated` were retained because missing update information itself can be meaningful.

- Created additional features:
  - `estimated_installs`
  - `update_year`
  - `update_month`
  - `days_since_update`
  - `update_missing`

- Converted date columns into appropriate datetime formats.

---

## 2. Country Dataset (`country_clean`)

The country-level dataset was cleaned while preserving its relationship with the app dataset.

Steps performed:

- Standardized column names.

- Removed duplicate records where required.

- Checked relationship with `apps_clean` using `app_id`.
  - All country records successfully matched with the app dataset.

- Missing app-level information:
  - `score`
  - `ratings`
  - `reviews`
  - `free`
  - `price`
  - `installs`
  - `min_installs`

  were recovered from `apps_clean` using `app_id` instead of statistical imputation because these attributes belong to the application itself and should remain consistent across countries.

- Removed temporary helper columns created during merging.

- Converted date columns into proper datetime format.

---

## 3. Discovery Dataset (`discovery_clean`)

The discovery dataset represents how applications were discovered through charts, searches, and archives.

Cleaning steps:

- Standardized column names.

- Preserved the `id` column because it can act as a primary key during SQL database creation.

- Checked relationships with `apps_clean` using `app_id`.
  - Most discovery records matched with the app dataset.
  - Unmatched records were retained because they represent valid discovery events and removing them could result in loss of historical discovery information.

- Handled missing values based on their meaning:
  - `country` missing values were replaced with `"Unknown"` because some discovery events were not country-specific.
  - `category`, `keyword`, and `collection` missing values were replaced with `"Not Available"` because these fields depend on the discovery source.
  - `chart_rank` missing values were retained because not every discovery source provides ranking information.

- Created additional analytical features:
  - `chart_rank_missing`
  - `discovery_year`
  - `discovery_month`
  - `discovery_hour`

- Converted `discovered_at` into datetime format.

---

## Final Dataset Status

Final cleaned datasets:

| Dataset | Rows | Columns |
|---|---:|---:|
| `apps_clean` | 11,176 | 38 |
| `country_clean` | 111,307 | 15 |
| `discovery_clean` | 75,812 | 13 |

All three datasets are now prepared for:
- SQL database creation
- Relational modelling
- Power BI dashboard development
- Exploratory data analysis